# Forecast Columns Explained: How to Defend Every Input and Every Exclusion

`06_forecast_column_selection.ipynb` records, for each of the 44 decidable columns, four test answers and a use (`input`, `provisional input` or `exclude`). This notebook explains **every one of those decisions** and prepares you for the questions a reviewer is likely to ask about them.

> **AI-assisted.** The explanations below were drafted with an AI assistant from the dataset paper and the project rules. Read them critically and defend the decisions in your own words. If you disagree with a decision, change it in notebook 06; the last cell here will then tell you which explanations are out of date.

## Natural-language request

> Create a notebook explaining why each and every column was accepted or rejected. Assume the person will be asking every sort of question to check why I have chosen a column.

## How to use this notebook

1. Run the cells in order (**Shift+Enter**). No cell reads `is_canceled`, `reservation_status` or `reservation_status_date` values.
2. Read **Part 1** first. Most challenges to a single column are really challenges to one of its principles.
3. **Part 2** is a one-page scoreboard: every column, its verdict and a one-line reason. Use it for quick answers.
4. **Parts 3 to 11** go group by group. Each column has the same layout:
   - **Verdict**: its four test answers and its use.
   - **What the paper says**: the source definition, quoted with the page number.
   - **Why**: the reasoning, in a few sentences.
   - **Challenges**: questions a reviewer is likely to ask, each with an answer.
5. **Part 12** covers the hardest cross-cutting questions. **Part 13** lists what evidence would change each excluded column's verdict.
6. The **final cell** checks that the decisions in notebook 06 still match the explanations written here.

**Source:** Antonio, de Almeida and Nunes (2019), "Hotel booking demand datasets", *Data in Brief* 22, 41–49 (`docs/1-s2.0-S2352340918315191-main.pdf`). Page numbers are the journal's (41–49). Table 1 (the variable definitions) runs over pp. 43–45.

## Part 1. The principles every decision rests on

**P1. The forecast moment.** On **morning M**, for **one hotel** and **one arrival night D** between M+1 and M+14, we forecast for **each booking on the book for D that morning** whether it will cancel. A column may be an input only if the hotel *has* its value on morning M **and** the value in our data *is* the value it had then.

**P2. The four tests.**

| Test | Question |
|---|---|
| T1 Not the outcome | Is the value something other than the result, and not written *because of* a cancellation, no-show, check-in or check-out? |
| T2 Known that morning | Does the hotel have the value on morning M? |
| T3 Will not change later | Is the value in our data the same as on morning M? |
| T4 This booking only | Is it built from this booking (and its guest's earlier history), not by comparing it with other rows of the file? |

**P3. "Unsure" counts as a failure.** If the paper does not let us answer a test, the column is excluded, and the reason names what would have to be verified. The burden of proof is on *including* a column, not on excluding it. A wrongly included leaky column makes every later result untrustworthy; a wrongly excluded clean column only costs some accuracy.

**P3a. The one exception: provisional inputs.** The team decided to use four columns with an "unsure" answer anyway: `is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled` and `company`. Their "unsure" answers and reasons are unchanged; only their use is `provisional input`. A column with any "no" answer can never be provisional. This gives up the protection of P3 for the information these columns carry, so every result that uses them must say that their timing is unverified, and Part 13 lists the evidence that would settle each one.

**P4. Decide by logic, never by predictive power.** A leaky column predicts *well* on train, on choose and on proof, because all three come from the same final export. A with/without test on choose cannot find leakage; it rewards it. The only defence is to reason about when each value is known. That is why notebooks 06 and 07 never read an outcome.

**P5. Why T3 is the test that bites.** The paper took each value from the change log "with respect to the day prior to the arrival" (p. 45). A booking that is cancelled cannot change after cancelling, so:
- a **kept** booking carries its value as of the day before arrival;
- a **cancelled** booking carries its value as of its cancellation.

For any amendable column, kept and cancelled bookings are frozen at *different moments*, and the moment depends on the outcome. The model could learn the outcome from that difference. The authors say so themselves: "it is understandable that the distribution of some variables differ between non canceled and canceled bookings" (p. 45).

**P6. Which attributes the paper says are amendable.** The paper names them in two places, and this is the main evidence used for T3:
- `BookingChanges` counts changes to "persons, arrival date, nights, reserved room type or meal" (p. 43).
- The word of caution: customers commonly change "the number of persons, staying duration, or room type preferences, either at the time of their check-in or during their stay", and hotels often do not know "the correct nationality of the customer until the moment of check-in" (p. 45).

Columns that **accumulate or are recalculated** up to arrival or cancellation (`booking_changes`, `total_of_special_requests`, `deposit_type`, `adr`) fail T3 by their own definitions.

**P7. "BL" in the source column is not, by itself, a failure.** Almost every variable lists "BO and BL" as its source, including `lead_time` and the arrival-date parts. That only means the authors *looked* in the change log for a day-before-arrival value. If "BL" alone failed T3, almost no column could be used. The test is whether the paper says the attribute is *amended*, *accumulated* or *set later*.

**P8. Built columns inherit.** A column built in notebook 05 (`nights`, `guests`, every `flag_*`) can be no cleaner than the columns it is made from. Notebook 06's final cell enforces this.

**P9. Scoring use is not input use.** `hotel` and `arrival_date` (grouping keys) and `adr` and `nights` (stay value) are used by `src/contract.py` whatever we decide (project rule 1). Excluding `adr` or `nights` as *inputs* does not remove them from the scoring.

**P10. What this cannot claim (project rule 7).** Our data is the final export, not a morning snapshot. Every "yes" on T3 is a judgment from the source definitions, not a proof. The scoring also counts each night's full final book, including bookings made after M (the **full-book caveat**). No column choice fixes that; timestamped reservation snapshots would.

## Setup

Loads the cleaned table from notebook 05 and **the decisions exactly as entered in notebook 06**, by running only notebook 06's `decide-*` cells. If you edit a decision there, this notebook picks it up. Evidence uses train and choose rows only, and no outcome column is ever shown.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "contract.py").exists():
        repo_root = folder
        break
if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import contract

clean_path = repo_root / "data" / "hotels_clean.pkl"
if not clean_path.exists():
    raise FileNotFoundError("data/hotels_clean.pkl is missing. Run notebooks/05_apply_cleaning_decisions.ipynb first.")
cleaned = pd.read_pickle(clean_path)
train, choose, proof = contract.split_periods(cleaned)
train_choose = pd.concat([train, choose])

outcome_columns = ["is_canceled", "reservation_status", "reservation_status_date"]
to_decide = [column for column in cleaned.columns if column != "is_canceled"]

# Load the decisions from notebook 06 by running only its decision cells.
notebook_06 = json.loads((repo_root / "notebooks" / "06_forecast_column_selection.ipynb").read_text(encoding="utf-8"))
my_decisions = {}
for cell in notebook_06["cells"]:
    if cell.get("id", "").startswith("decide-"):
        exec("".join(cell["source"]))

missing = [column for column in to_decide if column not in my_decisions]
if missing:
    raise ValueError(f"Notebook 06 has no decision for: {missing}")


def verdict(columns):
    # Test answers and use from notebook 06, plus value evidence on train + choose (never an outcome).
    rows = []
    for column in columns:
        answer = my_decisions[column]
        row = dict(zip(["T1", "T2", "T3", "T4"], answer["tests"]), use=answer["use"])
        if column not in outcome_columns:
            values = train_choose[column]
            row["distinct"] = values.nunique()
            row["most common values (rows)"] = ", ".join(f"{v} ({n:,})" for v, n in values.value_counts().head(4).items())
        rows.append({"column": column, **row})
    display(pd.DataFrame(rows).set_index("column"))


inputs = [column for column in to_decide if my_decisions[column]["use"] in ("input", "provisional input")]
print(f"{len(to_decide)} columns decided; {len(inputs)} inputs: {inputs}")

## Part 2. Scoreboard: every column in one line

**Inputs that pass all four tests (4).**

| Column | One-line reason |
|---|---|
| `hotel` | Fixed when the booking is made; no booking moves between hotels. |
| `market_segment` | How the booking was sold; not among the attributes the paper says are amended. |
| `distribution_channel` | How the booking arrived; not among the amended attributes. |
| `agent` | The agency that *made* the booking; fixed by the act of booking. |

**Provisional inputs (4).** Used by team decision although a test is unsure (P3a). Every result that uses them must say their timing is unverified.

| Column | What is unsure |
|---|---|
| `is_repeated_guest` | When the guest profile is linked to the booking. |
| `previous_cancellations` | Whether earlier bookings cancelled *after* morning M are counted. |
| `previous_bookings_not_canceled` | Whether an earlier booking open on M but cancelled later moves between counts. |
| `company` | Whether the paying company can be attached later, e.g. at billing. |

**Pass all four tests but not used (6).**

| Column | One-line reason |
|---|---|
| `arrival_date` | It is the grouping key; as a raw date, proof (2017) lies outside the training range. |
| `arrival_date_year` | Proof is 2017, a year the model never sees; not a leakage problem. |
| `arrival_date_month`, `arrival_date_week_number`, `arrival_date_day_of_month` | Describe the night being forecast, so they pass; not in the team's feature set. Can be added back on choose. |
| `lead_time` | The entry date is fixed, so it passes; not in the team's feature set. The strongest candidate to add back on choose. |

**Excluded as "unsure" (4).** Could become inputs if a specific fact is verified (Part 13).

| Column | What is unsure |
|---|---|
| `flag_repeat_status_conflict` | Built from the three history columns; the team did not choose it as provisional. |
| `days_in_waiting_list` | Whether a booking still waitlisted on M is on the book, and how never-confirmed bookings are coded. |
| `required_car_parking_spaces` | Whether parking is requested when booking or registered at check-in. |
| `customer_type` | Whether links to other bookings (Transient-party, Group) can form after M. |

**Excluded because the value is not known on morning M or changes after it (T2 or T3 = no) (23).**

| Column(s) | Reason |
|---|---|
| `stays_in_weekend_nights`, `stays_in_week_nights`, `nights` | Paper: staying duration is commonly changed, even at check-in. |
| `adults`, `children`, `babies`, `guests` | Paper: the number of persons is commonly changed. |
| `flag_zero_nights`, `flag_zero_guests`, `flag_children_without_adults`, `flag_ten_plus_adults`, `flag_children_missing` | Built from the columns above. |
| `meal`, `reserved_room_type`, `flag_room_type_p` | Named in `booking_changes` as amended attributes. |
| `total_of_special_requests` | A running total that grows until arrival. |
| `country` | Paper: correct nationality often known only at check-in. |
| `deposit_type` | Recalculated from payments up to arrival or cancellation. |
| `booking_changes` | Counts amendments up to check-in or cancellation. |
| `assigned_room_type` | Can be the *result* of overbooking (also fails T1). |
| `adr`, `flag_adr_above_1000`, `flag_adr_negative` | Built from lodging transactions, mostly posted during the stay. |

**Excluded as the outcome or a cross-row comparison (3).**

| Column | Reason |
|---|---|
| `reservation_status` | It *is* the outcome. Project rule 3. |
| `reservation_status_date` | The date of the outcome. Project rule 3. |
| `flag_identical_group` | Compares each row with every row in the file, including future and proof bookings. |

`is_canceled` is the **label**: no decision. Total: 4 + 4 + 6 + 4 + 23 + 3 = 44 columns.

## Part 3. Group A: the night being forecast

`hotel`, `arrival_date`, `arrival_date_year`, `arrival_date_month`, `arrival_date_week_number`, `arrival_date_day_of_month`

In [ ]:
verdict(["hotel", "arrival_date", "arrival_date_year", "arrival_date_month",
         "arrival_date_week_number", "arrival_date_day_of_month"])

# Which arrival years each period contains (dates only; no outcome read).
for name, period in {"train": train, "choose": choose, "proof": proof}.items():
    print(f"{name:>6}: arrival years {sorted(int(y) for y in period['arrival_date_year'].unique())}, "
          f"{period['arrival_date'].min():%Y-%m-%d} to {period['arrival_date'].max():%Y-%m-%d}")

### `hotel`: INPUT

**Verdict.** yes / yes / yes / yes, **input**.

**What the paper says.** H1 is a resort hotel in the Algarve and H2 a city hotel in Lisbon, extracted as two separate datasets from each hotel's own PMS (pp. 41–42). `hotel` is not one of the 31 Table 1 variables; it is the label that separates the two files.

**Why.** The property is decided when the booking is made and is certain on morning M. The paper never describes a booking moving between the two hotels, which are separate systems. Nothing about it is written because of the outcome, and it describes this booking alone.

**Challenges.**
- *The hotels have very different cancellation rates. Isn't `hotel` just a proxy for the outcome?* No. A column that **predicts** the outcome is fine; a column whose **value is caused by** the outcome is leakage. The hotel is fixed before anything happens. A higher rate at one hotel is exactly the kind of real pattern the model should learn.
- *It is already a grouping key. Why is it also an input?* The grouping key decides *which night's book* a booking belongs to; as an input it lets the model learn that the two hotels behave differently. The contract uses it as a key whatever we decide (P9).
- *Could a guest cancel at one hotel and rebook at the other?* That would be two bookings, each with a fixed hotel, not one booking whose hotel changed.

### `arrival_date_month`, `arrival_date_week_number`, `arrival_date_day_of_month`: PASS, NOT USED

**Verdict.** yes / yes / yes / yes, **exclude** (all three): they pass, but are not in the team's feature set.

**What the paper says.** Month, week number and day of month "of the arrival date", from "BO and BL" (p. 43). Arrival date is one of the attributes `BookingChanges` lists as amended (p. 43).

**Why.** On morning M we forecast a **specific night D**, so D's month, week and day are certain and can never change. These values describe the **night**, not the booking. An amended arrival date does not change the month of night D; it moves the booking to a *different* night in our data. Which bookings sit on which night is a question about the book (the full-book caveat, P10), not about the value of this column for night D.

**Challenges.**
- *They pass every test. Why are they not used?* The team fixed the inputs to eight columns (Part 2). Passing makes a column *allowed*, not *required* (see `arrival_date` below). Adding them back is a modelling choice to test on choose, never on proof (project rule 2).
- *In notebook 01 you put these in bin 2 (amendable). Why do they pass now?* Notebook 01 asked "is the value fixed at booking?" and the answer is no: a guest can move the arrival. Notebook 06 asks "is the value known and unchanged *for the night being forecast*?" For night D the answer is yes. The question changed, so the answer changed.
- *Isn't the arrival-date amendment still leakage? Kept bookings can move, cancelled ones cannot move after cancelling.* That is a real effect, but it changes **which bookings appear on which night** in our data, for every column alike. Dropping the month would not remove it. It belongs to the full-book caveat, which needs timestamped snapshots to fix (project rule 7), and it is reported as a limitation, not hidden.
- *Month and week number say almost the same thing. Why keep both?* Both pass the leakage tests. Whether the model needs both is a **modelling** question, to settle on choose (project rule 2), not a leakage question.
- *Does the day of the month carry any information?* Possibly little (pay-days, month-end events). Again a modelling question: passing the tests makes it *allowed*, not *required*.
- *Why not day of week?* It is not a column of the cleaned table. It could be derived from `arrival_date` and would pass the same tests; adding it is a feature-engineering step for later, not a decision this notebook makes.

### `arrival_date`: PASSES, NOT USED

**Verdict.** yes / yes / yes / yes, **exclude**.

**Why.** It passes for the same reason as the month: it *is* night D. It is kept out of the model for two non-leakage reasons:
1. It is the contract's **grouping key**; the scoring uses it anyway (P9).
2. As a raw date the model would see train and choose dates (2015-07 to 2016-12) and then be asked about proof dates (2017), all outside the range it learned from.

The month, week and day would carry its seasonal information in a form that repeats every year; they also pass, but are not in the team's current feature set.

**Challenges.**
- *You said a column is excluded only when it fails a test. Isn't excluding a passing column inconsistent?* No. All four "yes" answers make a column **eligible**, they do not force it in. Notebook 06's check only forbids the reverse: an input that failed a test.
- *Did you exclude it because it performed badly?* No outcome was read. The reason is structural: proof dates are outside the training range, which is visible from the dates alone (cell above).

### `arrival_date_year`: PASSES, NOT USED

**Verdict.** yes / yes / yes / yes, **exclude**.

**Why.** The year of night D is certain on morning M, so it passes. But the evidence cell shows train holds 2015–2016, choose only 2016, and proof only **2017**. The model would be asked about a year it has never seen. A tree model would treat 2017 like 2016 (it cannot extrapolate); a linear model would extend whatever 2015–2016 trend it found. Neither is reliable.

**Challenges.**
- *Isn't there a real trend across years that the model should capture?* Maybe, but one and a half years of training cannot estimate a trend reliably, and choose (2016 only) cannot test it. If a trend matters, a separately justified adjustment would be safer than a raw year input.
- *Is this leakage?* No. It is an **out-of-range** problem. The reason column in notebook 06 says so explicitly.

## Part 4. Group B: the guest's earlier history

`is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`, `flag_repeat_status_conflict`

All four are **unsure on T3**. The three history columns are **provisional inputs** by team decision (P3a); the flag built from them stays excluded. This is the group where a reviewer is most likely to say "history is in the past, how can it leak?", so the answer needs to be precise.

In [ ]:
verdict(["is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled", "flag_repeat_status_conflict"])

# Do the three history columns agree with each other? (feature values only)
no_repeat_flag_but_history = (train_choose["is_repeated_guest"].eq(0)
                              & (train_choose["previous_cancellations"].gt(0)
                                 | train_choose["previous_bookings_not_canceled"].gt(0)))
print(f"is_repeated_guest = 0 but a prior count > 0: {no_repeat_flag_but_history.sum():,} train + choose rows "
      f"({no_repeat_flag_but_history.mean():.1%})")

### `is_repeated_guest`: PROVISIONAL INPUT (unsure)

**Verdict.** yes / yes / **unsure** / yes, **provisional input**.

**What the paper says.** "Variable created by verifying if a profile was associated with the booking customer. If so, and if the customer profile creation date was prior to the creation date for the booking on the PMS database it was assumed the booking was from a repeated guest." Source: BO, BL and C (p. 44).

**Why.** The comparison itself uses two dates fixed before the booking exists, so that part is clean. But the variable first requires that "a profile was associated with the booking", and the paper never says **when** that association is made. If a profile can be linked later (an amendment, or identification at check-in), a kept booking could turn from 0 to 1 while a cancelled booking never gets the chance. Under P3, "the paper does not say" means unsure, which would mean exclude. The team uses it provisionally instead (P3a), so this doubt travels with every result that uses it.

**Challenges.**
- *The definition literally says "prior to the creation date". Isn't that proof it is booking-time information?* It proves the **profile** existed before the booking. It does not say the **link** between profile and booking existed on morning M.
- *Do you have any evidence the link is unreliable?* The cell above shows rows where `is_repeated_guest` is 0 yet the guest has earlier bookings counted. The three history columns do not agree with each other, which is consistent with the link being made, or read, at different times. It does not prove leakage, but it removes the confidence a "yes" needs.
- *If you are unsure, why use it?* The team judged its information worth the risk. The "unsure" answer is kept so the risk stays visible. One look at change-log timestamps for profile links would settle it (Part 13).

### `previous_cancellations`: PROVISIONAL INPUT (unsure)

**Verdict.** yes / yes / **unsure** / yes, **provisional input**.

**What the paper says.** "In case there was no customer profile associated with the booking, the value is set to 0. Otherwise, the value is the number of bookings with the same customer profile **created before the current booking and canceled**." (p. 44)

**Why.** The definition limits when earlier bookings were **created**. It does not limit when they were **cancelled**, and it does not say as of which date their status was read. Example: a guest makes booking A, then booking B, and cancels both on the same day after morning M. If the status is read from the final data, B's `previous_cancellations` counts A's cancellation, which happened at the same moment as B's own. The count would then carry B's outcome. The team uses it provisionally anyway (P3a); this example is the risk that must be reported with any result that uses it.

**Challenges.**
- *Isn't that example far-fetched?* Guests and agents cancelling several bookings at once is common (a trip falls through, a group cancels). The point is not how often it happens, but that the definition does not rule it out and our data has no timestamps to check.
- *Notebook 05 (decision C6) planned to test this column with and without on choose. Why not just do that?* Because choose labels come from the same final export. If the column leaks, it leaks on choose too and looks *better*, not worse (P4). A with/without test measures usefulness, not timing. That is why it is only **provisional**: a better choose score with it would not show it is clean.
- *T4 says "this booking only", but this column looks at other bookings. Why is T4 "yes"?* T4 allows "its guest's earlier history". The problem here is not *which* bookings are counted but *when* their status was read, which is T3.

### `previous_bookings_not_canceled`: PROVISIONAL INPUT (unsure)

**Verdict.** yes / yes / **unsure** / yes, **provisional input**.

**What the paper says.** "...the number of bookings with the same customer profile created before the current booking and not canceled." (p. 44)

**Why.** The mirror image of `previous_cancellations`. An earlier booking that has not yet arrived on morning M counts as "not cancelled" that morning. If it cancels later, the final data moves it into the other count. Same unanswered question: status as of when?

**Challenges.**
- *Does an earlier booking that has already been completed (checked out) cause any problem?* No. That status is final before M. The problem is only earlier bookings still open on M; the paper does not tell us how to tell them apart.

### `flag_repeat_status_conflict`: EXCLUDED (inherits)

**Verdict.** yes / yes / **unsure** / yes, **exclude**.

**Why.** Built in notebook 05 only from the three columns above, all unsure on T3. A flag cannot be cleaner than its sources (P8).

**Challenges.**
- *The flag only marks a data inconsistency. Isn't that harmless?* The inconsistency itself may come from the timing problem described above, so the flag can carry the same leak.
- *Its three sources are provisional inputs. Why isn't the flag?* Being provisional is a team choice made column by column, not something a built column inherits. The flag was built to track a data inconsistency during cleaning, and the team did not choose it as an input.

## Part 5. Group C: when the booking was made

`lead_time` (passes, not used), `days_in_waiting_list` (excluded)

In [ ]:
verdict(["lead_time", "days_in_waiting_list"])

# Waiting list: how many bookings were confirmed only inside the 14-day forecast window?
waitlisted = train_choose["days_in_waiting_list"].gt(0)
days_confirmed_before_arrival = train_choose["lead_time"] - train_choose["days_in_waiting_list"]
late_confirmation = waitlisted & days_confirmed_before_arrival.le(14)
print(f"Bookings that spent time on the waiting list: {waitlisted.sum():,} ({waitlisted.mean():.1%})")
print(f"  of which confirmed 14 or fewer days before arrival: {late_confirmation.sum():,}")

# Lead time: how much of the final book was booked inside the 14-day window (full-book caveat)?
short = train_choose["lead_time"].le(14)
print(f"Bookings made 14 or fewer days before arrival: {short.sum():,} ({short.mean():.1%})")

### `lead_time`: PASSES, NOT USED

**Verdict.** yes / yes / yes / yes, **exclude**: it passes, but is not in the team's feature set.

**What the paper says.** "Number of days that elapsed between the entering date of the booking into the PMS and the arrival date." Source: BO and BL, "Subtraction of the entering date from the arrival date" (p. 44).

**Why.** Lead time is (arrival date) minus (entry date). The **entry date never changes**: a booking is entered once. A booking on the book on morning M has already been entered, so the hotel can compute D minus entry date. If the arrival date is later amended, the export measures to the new date, but the booking also sits on that new night in our data. So the row's lead time always matches the night it is scored on. The only moving part is the arrival date, already handled in Group A.

**Challenges.**
- *It passes every test and is a well-known strong predictor. Why leave it out?* The team fixed the inputs to eight columns (Part 2). It is the strongest candidate to add back, as a modelling choice tested on choose (project rule 2).
- *In notebook 01 you put it in bin 2 and said it "can be amended later". What changed?* Notebook 01 treated lead time as an amendable booking detail. Looking more closely, the only amendable part is the arrival date, and that is a question of *which night* the booking is on, not of the lead time for a given night. The entry date itself is fixed.
- *The source says "BO and BL". Doesn't BL mean it can change?* See P7. Nearly every column lists BL because the authors looked up day-before-arrival values. What matters is whether the attribute itself is amended; here only the arrival-date part is.
- *Lead time is famously one of the strongest cancellation predictors. Isn't that suspicious?* Being strong is not leakage. Bookings made far ahead have more time and more reason to cancel; that is real behaviour, known on the morning. What would be suspicious is a value *written because of* the outcome, and the entry date is not.
- *The evidence cell shows many bookings made within 14 days of arrival. Doesn't that matter?* Yes, but not for this column. On morning M, a booking with a lead time shorter than (D − M) did not exist yet. The scoring still counts it (full-book caveat, P10). That is a limitation of the whole exercise, reported separately; it does not make the lead time of an existing booking wrong.

### `days_in_waiting_list`: EXCLUDED (unsure)

**Verdict.** yes / **unsure** / **unsure** / yes, **exclude**.

**What the paper says.** "Number of days the booking was in the waiting list before it was confirmed to the customer." Source: BO only, calculated from the confirmation date and the entry date (p. 43).

**Why.** Once a booking is confirmed, the value is fixed. Two things are not known:
1. **Is a booking still on the waiting list on morning M "on the book" for night D?** The evidence cell shows bookings confirmed 14 or fewer days before arrival. On any morning before their confirmation, they were still waitlisted, and their final value did not yet exist (T2).
2. **How is a booking coded if it was cancelled before it was ever confirmed?** The paper does not say. If such bookings get a special value (for example days until cancellation), the value is shaped by the cancellation (T3).

**Challenges.**
- *The source is BO only, not BL. Doesn't that make it more reliable than the others?* It means it is not overwritten by change-log edits, which helps. It does not answer either question above.
- *Most bookings have 0. Why not keep it?* The zeros are probably fine; the problem is the few that are not, and those are exactly where the timing is unclear.

## Part 6. Group D: stay length and party size

`stays_in_weekend_nights`, `stays_in_week_nights`, `nights`, `adults`, `children`, `babies`, `guests`, and the flags `flag_zero_nights`, `flag_zero_guests`, `flag_children_without_adults`, `flag_ten_plus_adults`, `flag_children_missing`

All twelve are **excluded on T3 = no**. The paper explicitly names these attributes as ones that change.

In [ ]:
verdict(["stays_in_weekend_nights", "stays_in_week_nights", "nights", "adults", "children", "babies", "guests",
         "flag_zero_nights", "flag_zero_guests", "flag_children_without_adults", "flag_ten_plus_adults",
         "flag_children_missing"])

# How common are amendments at all? (booking_changes counts changes to persons, arrival date, nights, room type, meal)
amended = train_choose["booking_changes"].gt(0)
print(f"Bookings with at least one recorded amendment: {amended.sum():,} ({amended.mean():.1%})")

### `stays_in_weekend_nights`, `stays_in_week_nights`, `nights`: EXCLUDED

**Verdict.** yes / yes / **no** / yes, **exclude** (all three).

**What the paper says.** Weekend and week nights "the guest stayed or booked to stay", from BO and BL (p. 45). `BookingChanges` counts changes to "nights" (p. 43). Customers commonly change "staying duration ... either at the time of their check-in or during their stay" (p. 45).

**Why.** The hotel knows a stay length on morning M (T2 yes), but our data holds a **later** value: the day-before-arrival value for kept bookings, the at-cancellation value for cancelled ones (P5). The wording "stayed **or** booked to stay" itself shows the value can describe two different moments. `nights` is the sum of the two, so it inherits.

**Challenges.**
- *Stay length is basic booking information. Surely the hotel knows it?* It does know *a* value. T2 passes. T3 fails because the value in our data is not necessarily that one.
- *How often did the stay change after morning M?* We cannot measure it: the export has no timestamps. The evidence cell shows amendments of some kind are not rare. That is why this is a judgment from the definitions (P10).
- *The paper took values as of the day before arrival, so check-in changes should not be in the data. Doesn't that solve it?* Partly. Our forecast morning is up to 14 days before arrival, not one day. Changes between M and the day before arrival are still in. And cancelled bookings are frozen at cancellation, a different moment from kept ones. The authors themselves warn that the distributions differ between cancelled and kept bookings (p. 45).
- *`nights` is used in the contract's stay value. Isn't excluding it inconsistent?* No (P9). The contract uses it to put a euro value on an empty room. That is scoring, fixed by project rule 1. Using it as a model input is a separate question.

### `adults`, `children`, `babies`, `guests`: EXCLUDED

**Verdict.** yes / yes / **no** / yes, **exclude** (all four).

**What the paper says.** Number of adults, children ("sum of both payable and non-payable children") and babies, from BO and BL (p. 43). `BookingChanges` counts changes to "persons" (p. 43). Customers commonly change "the number of persons" (p. 45).

**Why.** The paper names party size as one of the attributes that change, including at check-in. Same freezing problem as stay length. `guests` is their sum and inherits.

**Challenges.**
- *Would the number of adults really differ between cancelled and kept bookings because of timing?* A kept booking can be updated right up to arrival (and the paper says often is); a cancelled one stops being updated when it cancels. Any update that happens close to arrival appears only on kept bookings.
- *Couldn't you keep `adults` and drop only `children`?* The paper's word is "persons", which covers all three. There is no source basis for treating adults differently.

### `flag_zero_nights`, `flag_zero_guests`, `flag_children_without_adults`, `flag_ten_plus_adults`, `flag_children_missing`: EXCLUDED

**Verdict.** The first four: yes / yes / **no** / yes. `flag_children_missing`: yes / **unsure** / **no** / yes. All **exclude**.

**Why.** All five were built in notebook 05 from stay or party-size columns that fail T3, so they inherit (P8). `flag_children_missing` has an extra problem: it marks a **blank in the final export**. The blank describes the export, not what the hotel saw that morning, so even T2 is unsure.

**Challenges.**
- *These flags mark data-quality problems, not booking behaviour. Why would they leak?* They are computed from values that change after M. A zero-night or zero-guest row may be zero *because* of what happened later (for example a booking emptied when it was cancelled). We cannot rule that out.
- *Then why were they built?* To find and track odd rows during cleaning (notebook 05), not to be model inputs.

## Part 7. Group E: what was booked, and the price

`meal`, `reserved_room_type`, `assigned_room_type`, `adr`, `required_car_parking_spaces`, `total_of_special_requests`, `flag_room_type_p`, `flag_adr_above_1000`, `flag_adr_negative`

All nine **excluded**.

In [ ]:
verdict(["meal", "reserved_room_type", "assigned_room_type", "adr", "required_car_parking_spaces",
         "total_of_special_requests", "flag_room_type_p", "flag_adr_above_1000", "flag_adr_negative"])

differs = train_choose["assigned_room_type"].ne(train_choose["reserved_room_type"])
print(f"assigned_room_type differs from reserved_room_type: {differs.sum():,} train + choose bookings ({differs.mean():.1%})")

### `meal`: EXCLUDED

**Verdict.** yes / yes / **no** / yes, **exclude**.

**What the paper says.** "Type of meal booked", from BO, BL and ML (p. 44). `BookingChanges` counts changes to "meal" (p. 43).

**Why.** The paper names meal as an amended attribute, so our value may be later than morning M.

**Challenges.**
- *Who changes their meal plan?* Upgrading to half board before arrival is a routine amendment. The paper counts it, which is all T3 needs.
- *Notebook 05 recoded `Undefined` to `SC`. Does that affect this?* No. The recode is about meaning (both mean "no meal package"), not about timing.

### `reserved_room_type` and `flag_room_type_p`: EXCLUDED

**Verdict.** yes / yes / **no** / yes, **exclude** (both).

**What the paper says.** "Code of room type reserved", from BO and BL (p. 45). `BookingChanges` counts changes to "reserved room type" (p. 43). Customers commonly change "room type preferences" (p. 45).

**Why.** Named twice as amendable. `flag_room_type_p` is built from it and inherits.

**Challenges.**
- *The reserved room type is what the guest booked. Isn't that booking-time information?* The *first* reserved type is. Our column holds the *latest* one.

### `assigned_room_type`: EXCLUDED

**Verdict.** **no** / **unsure** / **no** / yes, **exclude**.

**What the paper says.** "Sometimes the assigned room type differs from the reserved room type due to hotel operation reasons (e.g. **overbooking**) or by customer request." (p. 43)

**Why.** This is the clearest case after the outcome columns themselves.
- **T1 no.** An assignment changed because of overbooking is a **consequence** of the very decision this forecast feeds. The model would learn from the results of past overselling.
- **T3 no.** For a guest who stays, the final room is settled around arrival; a cancelled booking keeps whatever was assigned before it cancelled.
- **T2 unsure.** The paper does not say when a room is first assigned.

**Challenges.**
- *The evidence shows the assigned type differs from the reserved type on a sizeable share of bookings. Couldn't that just be guest upgrades?* Some will be. But we cannot separate those from overbooking moves, and the paper names overbooking explicitly.

### `adr`, `flag_adr_above_1000`, `flag_adr_negative`: EXCLUDED

**Verdict.** **unsure** / **no** / **no** / yes, **exclude** (all three).

**What the paper says.** "Calculated by dividing the sum of all **lodging transactions** by the total number of staying nights", from BO, BL and TR (p. 43).

**Why.** `adr` is not the rate quoted at booking; it is built from **transactions**.
- **T2 no.** For a guest who stays, lodging is normally posted night by night during the stay, after morning M.
- **T3 no.** It divides by staying nights, which fail T3, and transactions keep being posted until departure.
- **T1 unsure.** For a cancelled booking it is unclear which transactions exist at all. Whatever is there may be there because the booking was cancelled.

The two flags are built from `adr` and inherit.

**Challenges.**
- *Price is the most important economic variable. Isn't excluding it a big loss?* Probably, yes. The fix is a **rate snapshot**: the rate on the booking as of morning M. This column is not that.
- *The contract uses `adr`. Isn't excluding it inconsistent?* No (P9). The contract uses `adr` × `nights` to put a euro value on an empty room. That is scoring, fixed by project rule 1.
- *Negative and above-1,000 ADR values exist. Did that influence the decision?* No. Those are data-quality questions handled in notebook 05. The exclusion would stand even if every value were tidy.

### `required_car_parking_spaces`: EXCLUDED (unsure)

**Verdict.** yes / **unsure** / **unsure** / yes, **exclude**.

**What the paper says.** "Number of car parking spaces required by the customer", from BO and BL (p. 44). No statement about when it is recorded.

**Why.** The paper does not say whether parking is requested when booking or registered when the guest arrives by car. If it is often recorded at check-in, only guests who actually arrive can have a non-zero value, and the column would mark stays rather than predict them. The paper's caution that guests change attributes at check-in (p. 45) makes this plausible.

**Challenges.**
- *It is not in the paper's list of amended attributes. By P7, shouldn't it pass like `market_segment`?* The difference is that parking is a **guest request**, the kind of thing the caution paragraph says is updated at check-in, and it is something that only becomes concrete when the guest drives up. `market_segment` describes how the booking was sold, which happens once. This is a judgment; Part 13 says how to settle it.
- *If it turned out to be a very strong predictor, would that change your mind?* It would make me *more* suspicious, not less. A value that is almost always zero for cancelled bookings is exactly the signature of something recorded on arrival. (We have deliberately not looked.)

### `total_of_special_requests`: EXCLUDED

**Verdict.** yes / yes / **no** / yes, **exclude**.

**What the paper says.** "Number of special requests made by the customer (e.g. twin bed or high floor)", "Sum of all special requests" (p. 45).

**Why.** It is a **running total**. Requests can be added up to arrival. A kept booking keeps accumulating; a cancelled booking stops at cancellation. The total therefore differs partly *because of* the outcome.

**Challenges.**
- *Requests are usually made when booking.* Many are, but "sum of all" includes any added later, and our data cannot separate them.

## Part 8. Group F: who sold the booking, and who the guest is

`market_segment`, `distribution_channel`, `agent` (inputs); `company` (provisional input); `customer_type`, `country` (excluded)

This group holds the **least certain inputs**. Be ready to defend P7 here.

In [ ]:
verdict(["market_segment", "distribution_channel", "agent", "company", "customer_type", "country"])

# Agents in proof that never appear in train or choose (a modelling issue, not leakage).
seen_agents = set(train_choose["agent"])
unseen = ~proof["agent"].isin(seen_agents)
print(f"Proof rows whose agent never appears in train + choose: {unseen.sum():,} ({unseen.mean():.1%})")
print(f"Rows with country 'unknown' in train + choose: {train_choose['country'].eq('unknown').sum():,}")

### `market_segment`: INPUT

**Verdict.** yes / yes / yes / yes, **input**.

**What the paper says.** "Market segment designation", from BO, BL and MS (p. 44). It is **not** among the attributes `BookingChanges` counts (p. 43), and **not** in the caution paragraph (p. 45).

**Why.** The segment describes **how the booking was sold** (online agency, offline agency, group, direct, corporate). That happens once, when the booking is entered. The paper lists exactly which attributes are amended, twice, and segment is in neither list.

**Challenges.**
- *In notebook 01 you marked it "uncertain". Now it is "yes". What changed?* Notebook 01 had no specific test. Here the test is P6: the paper's own lists of amended attributes. Segment is not on them.
- *By your own rule, isn't "the paper does not say it can't change" just "unsure"?* The rule (T3) fails a column when the source says it **can** be amended, accumulates or is recalculated. If "the paper does not explicitly guarantee it is fixed" were enough to fail, every column would fail, including `lead_time` and `hotel`. P7 explains why "BL" alone is not the test.
- *The `Groups` segment sounds like `customer_type = Group`, which you excluded. Isn't that inconsistent?* They are defined differently. `customer_type` depends on **links to other bookings** that can be added later (Transient becomes Transient-party). `market_segment` is the sales channel recorded when the booking was taken. That said, this is the weakest of the four fully passing inputs: if the change log showed bookings being reclassified into `Groups` later, it would be excluded (Part 13).
- *Isn't a booking's segment a proxy for cancellation behaviour?* It predicts it, which is fine. It is not caused by it.

### `distribution_channel`: INPUT

**Verdict.** yes / yes / yes / yes, **input**.

**What the paper says.** "Booking distribution channel", from BO, BL and DC (p. 44). Not among the amended attributes.

**Why.** The channel through which the booking arrived (TA/TO, direct, corporate, GDS) is a fact of how it was created. Same reasoning as `market_segment`.

**Challenges.**
- *Channel and segment overlap heavily. Why keep both?* Both pass the leakage tests. Redundancy is a modelling question for choose (project rule 2).
- *A few rows are `Undefined`. Is that a problem?* A data-quality detail, not a timing one; a handful of rows.

### `agent`: INPUT

**Verdict.** yes / yes / yes / yes, **input**.

**What the paper says.** "ID of the travel agency that **made** the booking", from BO and BL (p. 43). `NULL` means "not applicable", i.e. no agency (p. 45); notebook 05 labelled it `none`.

**Why.** "Made the booking" ties the value to the act of booking. That happens before morning M and does not repeat. Not among the amended attributes.

**Challenges.**
- *Can a booking be transferred to another agency?* The paper does not mention it, and the definition is about who *made* it, not who manages it now. A change-log check would settle it (Part 13).
- *There are hundreds of agent IDs, and some proof agents never appear in train or choose (evidence cell). Is that a problem?* Yes, but a **modelling** one, not leakage. Rare and unseen agents need to be grouped (for example into "other"); that is decided on choose.
- *Isn't `agent` close to `company`, which is only a provisional input?* No. `agent` is who *made* the booking; `company` includes who *pays* for it, which can be settled later. That is why `agent` passes and `company` is only provisional. See below.

### `company`: PROVISIONAL INPUT (unsure)

**Verdict.** yes / **unsure** / **unsure** / yes, **provisional input**.

**What the paper says.** "ID of the company/entity that made the booking **or responsible for paying the booking**", from BO and BL (p. 43).

**Why.** The "or responsible for paying" half is the problem. A paying entity can be attached late, for example when the invoice is settled at check-out, and only guests who stay reach check-out. The paper does not say when the field is filled. The team uses it provisionally anyway (P3a).

**Challenges.**
- *Almost every row is `none`. What harm can it do?* The harm is in the small share that is filled. If those were filled mostly at billing, they would mark stays.
- *Then why is `agent` fine?* Its definition has no "responsible for paying" part; it is only who made the booking.

### `customer_type`: EXCLUDED (unsure)

**Verdict.** yes / **unsure** / **unsure** / **unsure**, **exclude**.

**What the paper says.** Contract, Group, Transient ("not part of a group or contract, and is not associated to other transient booking") and Transient-party ("transient, but is associated to at least other transient booking"), from BO and BL (p. 43).

**Why.** Transient versus Transient-party depends on **links to other bookings**. If a friend books a second room after morning M and the two are linked, the first booking turns from Transient into Transient-party. So:
- **T2/T3 unsure**: the value may not be the one held on M.
- **T4 unsure**: the value depends on other bookings, possibly ones made after M.

**Challenges.**
- *Isn't T4 about columns we build ourselves by comparing rows?* Mostly, but its purpose is to keep out values that depend on bookings the hotel did not yet have on M. A hotel-made link to another booking can do that too.
- *Contract bookings are surely known at booking time.* Probably, but the column mixes all four types, and the Transient/Transient-party split is the large part of it.

### `country`: EXCLUDED

**Verdict.** yes / **no** / **no** / yes, **exclude**.

**What the paper says.** "It is also common for hotels not to know the correct nationality of the customer until the moment of check-in." (p. 45)

**Why.** The paper states it directly. On morning M the correct value is often not known (T2 no). Guests who check in have it corrected; cancelled bookings never reach check-in and keep the original or unknown value (T3 no). The difference between them is created by the outcome.

**Challenges.**
- *The booking still has a country on morning M, even if it is not the corrected one. Why not use that?* Because our data does not hold that value; it holds the corrected one for guests who arrived.
- *Notebook 05 labelled missing values `unknown`. Does that make it safer?* No. It may make it worse: an `unknown` country can itself mean "never reached check-in".

## Part 9. Group G: payment and amendments

`deposit_type`, `booking_changes`, both **excluded**.

In [ ]:
verdict(["deposit_type", "booking_changes"])

### `deposit_type`: EXCLUDED

**Verdict.** **unsure** / **no** / **no** / yes, **exclude**.

**What the paper says.** "Value calculated based on the payments identified for the booking in the transaction (TR) table **before the booking's arrival or cancellation date**." No payment: No Deposit; payment at least equal to the total stay cost: Non Refund; otherwise Refundable (p. 43).

**Why.**
- **T2/T3 no.** It is recalculated from every payment up to arrival or cancellation. A payment made after morning M changes the value, and the payment window ends at a different moment for kept and cancelled bookings (P5).
- **T1 unsure.** If a charge posted *when* a booking is cancelled (a cancellation penalty) counts as a payment "before the cancellation date", the value would be written because of the cancellation.

**Challenges.**
- *The client's brief says prepaid bookings seem never to turn into stays. Isn't that exactly the signal we want?* It is a reason to **investigate** this column, not to use it. Economically, a guest who has paid the full stay up front should be the *least* likely to cancel. A value that behaves the opposite way is what you would expect if the "deposit" were often recorded at or because of cancellation. We have not checked the outcome here, and we cannot prove it either way without payment timestamps.
- *Hotels do know on the morning whether a deposit has been paid.* Yes, a **payment snapshot dated M** would be a fine input. This column is not that snapshot.

### `booking_changes`: EXCLUDED

**Verdict.** yes / **no** / **no** / yes, **exclude**.

**What the paper says.** "Number of changes/amendments made to the booking from the moment the booking was entered on the PMS **until the moment of check-in or cancellation**." (p. 43)

**Why.** The count includes changes after morning M (T2 and T3 no), and the counting window **ends at the outcome**: a cancelled booking stops counting at cancellation, a kept one keeps counting until check-in. Kept bookings have more time to collect changes.

**Challenges.**
- *Why is T1 "yes" if the window ends at the outcome?* T1 asks whether the value is the outcome or written because of it. Each change is a guest amendment, not the outcome. The outcome-dependent window is a T3 problem. Either way the column is excluded.
- *Couldn't you use "number of changes made before M"?* Yes, from a timestamped change log. That is not what this column holds.

## Part 10. Group H: the outcome columns

`reservation_status`, `reservation_status_date`, both **excluded**. Project rule 3 forbids them whatever the tests say. No values are shown.

In [ ]:
verdict(["reservation_status", "reservation_status_date"])  # decisions only; values are never shown

### `reservation_status` and `reservation_status_date`: EXCLUDED

**Verdict.** **no** / **no** / **no** / yes, **exclude** (both).

**What the paper says.** `ReservationStatus`: "Reservation last status", Canceled, Check-Out or No-Show (p. 44). `ReservationStatusDate`: "Date at which the last status was set ... to understand when was the booking canceled or when did the customer checked-out" (p. 45).

**Why.** The status *is* the outcome; the date is *when* the outcome happened. Neither exists on morning M.

**Challenges.**
- *Why answer T4 "yes"?* Each describes this booking only. T4 is not why they fail; T1 to T3 are. Answering honestly test by test, rather than marking everything "no", shows the tests were actually applied.
- *Could the status date be used just to measure how early cancellations happen?* As an **analysis** tool on train, yes. As a model **input**, never.
- *How does `No-Show` relate to the label?* The contract's `is_canceled` = 1 includes no-shows; the paper keeps No-Show as a separate status. That matters for the label, not for this decision.

## Part 11. Group I: the cross-row flag

`flag_identical_group`, **excluded**.

In [ ]:
verdict(["flag_identical_group"])

### `flag_identical_group`: EXCLUDED

**Verdict.** yes / **no** / **no** / **no**, **exclude**.

**Why.** Built in notebook 05 by comparing each row with **every other row in the whole file**, including train, choose and **proof**, across all non-outcome columns.
- **T1 yes**: it uses no outcome column.
- **T2 no**: on morning M the hotel does not have bookings made after M, let alone bookings in the proof period.
- **T3 no**: it compares final values of amendable columns (stay, persons, room, meal, deposit).
- **T4 no**: it is the textbook example of a value built from other rows.

**Challenges.**
- *It uses no outcome. How can it leak?* It leaks **future information**, not the label directly. Whether a booking has an identical twin depends on bookings that do not exist yet, and on final values. It also lets proof-period rows influence train features, which breaks project rule 2.
- *Then why was it built?* To investigate repeated rows during cleaning (notebooks 03 to 05), not as an input.

## Part 12. The hardest cross-cutting questions

**"Why didn't you just keep the columns that predict best?"**
Because a leaky column predicts best of all, on train, choose and proof alike, since all three come from the same final export (P4). It would then fail in real use, where the leaked information is not available on the morning. Choosing by logic is the only protection.

**"The paper's authors say they designed the data to prevent leakage. Why don't you trust them?"**
They did, for *their* question. They took values as of **the day before arrival** (p. 42, p. 45). Our forecast is made up to **14 days** before arrival, so changes in between are in the data. Cancelled bookings are frozen at cancellation, a different moment from kept ones. And the authors themselves warn that distributions differ between cancelled and kept bookings for that reason (p. 45).

**"You threw away most of the columns. Won't the model be weak?"**
It may be weaker than a leaky one would *appear* to be. That is the point: the apparent accuracy of a leaky model is not real. Project rule 5 requires every candidate to be compared with the flat-rule baseline on the same hotel-nights, in euros, empty rooms and walked guests. If these eight columns (four of them provisional) cannot beat the baseline, that is a finding worth reporting, and the remedy is timestamped snapshots, not leaky columns.

**"Could you add columns back after seeing results on choose?"**
Not the ones excluded for timing: choose cannot detect timing leakage (P4). The same limit applies to the four provisional inputs: a good choose score does not make them clean. Choose *is* the place for modelling decisions among the inputs and the passing columns not yet used (adding back `lead_time` or the arrival-date parts, redundant columns, grouping rare agents, adding a derived weekday), per project rule 2. Proof is still run once.

**"Did you look at cancellation outcomes while deciding?"**
No. Notebooks 06 and 07 never read `is_canceled`, `reservation_status` or `reservation_status_date` values; the evidence helper refuses to show them.

**"Does careful column choice fix the full-book caveat?"**
No. They are separate problems. Column choice stops leakage through **feature values**. The full-book caveat is about **which bookings are counted**: the scoring counts each night's final book, including bookings made after morning M and bookings whose arrival moved. Both are reported as limitations; only timestamped snapshots fix either completely.

**"Can you claim this works as a live morning forecast?"**
No (project rule 7). Every "yes" on T3 is a judgment from definitions, and the scored book is the final book. A live forecast needs timestamped reservation snapshots.

**"Your notebook 01 bins and these decisions disagree in places. Which is right?"**

| Column(s) | Notebook 01 bin | Now | Why the answer changed |
|---|---|---|---|
| `arrival_date_month`, `_week_number`, `_day_of_month` | 2 | passes, not used | Notebook 01 asked about the booking; here they describe the forecast night, which is certain. Not in the team's current feature set. |
| `lead_time` | 2 | passes, not used | The only amendable part is the arrival date (a which-night question); the entry date is fixed. Not in the team's current feature set. |
| `market_segment`, `distribution_channel`, `agent` | 2 ("uncertain") | input | Checked against the paper's explicit lists of amended attributes (P6); not on them. |
| `is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled` | 1 | provisional input (unsure) | "Historical" was too quick: the link and the status read-date are not specified. |
| `arrival_date_year` | 2 | passes, not used | Passes the tests; excluded for an out-of-range year, not timing. |

The two notebooks ask different questions ("fixed at booking?" versus "known and unchanged on the forecast morning?"), so some answers should differ.

**"Which of your decisions are you least sure about?"**
1. The four provisional inputs (`is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`, `company`): used although a test is unsure. This is the largest open risk in the input set.
2. `market_segment`, `distribution_channel`, `agent`: included because the paper does not list them as amended. That is absence of evidence, judged acceptable under P7.
3. The Group A reasoning that an amended arrival only moves a booking between nights: sound for the column values, but the movement itself is a real limitation (full-book caveat).

## Part 13. What evidence would change each verdict

With a **timestamped change log** or **morning snapshots**, these checks would settle each open question:

| Column | Verdict now | Evidence that would change it |
|---|---|---|
| `market_segment`, `distribution_channel`, `agent` | input | Change-log entries reclassifying them after booking → **exclude**. |
| `is_repeated_guest` | provisional input | Profile link always made at booking creation → **input**; often made later → **exclude**. |
| `previous_cancellations`, `previous_bookings_not_canceled` | provisional input | Counts recomputed using only statuses as of morning M → **input**; counts shown to include later statuses → **exclude**. |
| `lead_time`, `arrival_date_month`, `_week_number`, `_day_of_month` | passes, not used | No new evidence needed: adding them back is a modelling choice, tested on choose. |
| `days_in_waiting_list` | unsure | Waitlisted bookings shown as not on the book, and never-confirmed coding known → **input**. |
| `required_car_parking_spaces` | unsure | Parking always requested at booking, not at check-in → **input**. |
| `company` | provisional input | Company field always set at booking → **input**; often set at billing → **exclude**. |
| `customer_type` | unsure | Links to other bookings only made when this booking is created → **input**. |
| Stay, persons, meal, room type, requests | no | Values taken from a snapshot dated morning M → **input** (as snapshot columns). |
| `adr` | no | The booked rate as of morning M (a rate snapshot) → a new, clean **input**. |
| `deposit_type` | no | Payments received by morning M (a payment snapshot) → a new, clean **input**. |
| `booking_changes` | no | Changes made before morning M only → a new, clean **input**. |
| `country` | no | Nationality as recorded on morning M → a new **input**; the corrected value never. |
| `assigned_room_type`, outcome columns, `flag_identical_group` | no | Nothing: they are consequences of the outcome or of future data. |

## Final cell: are these explanations still up to date?

The explanations above were written for specific test answers. This cell compares them with the decisions currently in notebook 06 and lists any column whose explanation no longer matches. Re-read and rewrite those sections if anything is listed.

In [ ]:
Y, N, U = "yes", "no", "unsure"
explained = {
    "hotel": ([Y, Y, Y, Y], "input"),
    "arrival_date": ([Y, Y, Y, Y], "exclude"),
    "arrival_date_year": ([Y, Y, Y, Y], "exclude"),
    "arrival_date_month": ([Y, Y, Y, Y], "exclude"),
    "arrival_date_week_number": ([Y, Y, Y, Y], "exclude"),
    "arrival_date_day_of_month": ([Y, Y, Y, Y], "exclude"),
    "is_repeated_guest": ([Y, Y, U, Y], "provisional input"),
    "previous_cancellations": ([Y, Y, U, Y], "provisional input"),
    "previous_bookings_not_canceled": ([Y, Y, U, Y], "provisional input"),
    "flag_repeat_status_conflict": ([Y, Y, U, Y], "exclude"),
    "lead_time": ([Y, Y, Y, Y], "exclude"),
    "days_in_waiting_list": ([Y, U, U, Y], "exclude"),
    "stays_in_weekend_nights": ([Y, Y, N, Y], "exclude"),
    "stays_in_week_nights": ([Y, Y, N, Y], "exclude"),
    "nights": ([Y, Y, N, Y], "exclude"),
    "adults": ([Y, Y, N, Y], "exclude"),
    "children": ([Y, Y, N, Y], "exclude"),
    "babies": ([Y, Y, N, Y], "exclude"),
    "guests": ([Y, Y, N, Y], "exclude"),
    "flag_zero_nights": ([Y, Y, N, Y], "exclude"),
    "flag_zero_guests": ([Y, Y, N, Y], "exclude"),
    "flag_children_without_adults": ([Y, Y, N, Y], "exclude"),
    "flag_ten_plus_adults": ([Y, Y, N, Y], "exclude"),
    "flag_children_missing": ([Y, U, N, Y], "exclude"),
    "meal": ([Y, Y, N, Y], "exclude"),
    "reserved_room_type": ([Y, Y, N, Y], "exclude"),
    "assigned_room_type": ([N, U, N, Y], "exclude"),
    "adr": ([U, N, N, Y], "exclude"),
    "required_car_parking_spaces": ([Y, U, U, Y], "exclude"),
    "total_of_special_requests": ([Y, Y, N, Y], "exclude"),
    "flag_room_type_p": ([Y, Y, N, Y], "exclude"),
    "flag_adr_above_1000": ([U, N, N, Y], "exclude"),
    "flag_adr_negative": ([U, N, N, Y], "exclude"),
    "market_segment": ([Y, Y, Y, Y], "input"),
    "distribution_channel": ([Y, Y, Y, Y], "input"),
    "agent": ([Y, Y, Y, Y], "input"),
    "company": ([Y, U, U, Y], "provisional input"),
    "customer_type": ([Y, U, U, U], "exclude"),
    "country": ([Y, N, N, Y], "exclude"),
    "deposit_type": ([U, N, N, Y], "exclude"),
    "booking_changes": ([Y, N, N, Y], "exclude"),
    "reservation_status": ([N, N, N, Y], "exclude"),
    "reservation_status_date": ([N, N, N, Y], "exclude"),
    "flag_identical_group": ([Y, N, N, N], "exclude"),
}

stale = []
for column in to_decide:
    if column not in explained:
        stale.append(f"{column}: no explanation written in this notebook")
        continue
    tests, use = explained[column]
    current = my_decisions[column]
    if current["tests"] != tests or current["use"] != use:
        stale.append(f"{column}: explained as {tests} / {use}, notebook 06 now says {current['tests']} / {current['use']}")
stale += [f"{column}: explained here but not a column of the cleaned table" for column in set(explained) - set(to_decide)]

if stale:
    print(f"{len(stale)} explanation(s) out of date:")
    for item in stale:
        print("  -", item)
else:
    print(f"All {len(to_decide)} explanations match the decisions in notebook 06.")
    print(f"Inputs ({len(inputs)}): {inputs}")